# coleman_coalitions — minimal working example

This notebook walks through the main steps of a Coleman coalition analysis:
1. Load (or define) inputs
2. Run the full Coleman analysis
3. Run coalition analysis
4. Visualise results

**Install first:** `pip install -e .` (from the repo root) or `pip install numpy matplotlib networkx`

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import coleman_coalitions as cc

## 1. Load a built-in dataset

Three canonical examples from Coleman (1973) are included:

| Function | Actors | Events | Resources | Notes |
|---|---|---|---|---|
| `standard_variables_trad12` | 3 | 4 | 4 | Identity resource matrix, mixed-sign interests |
| `standard_variables_techno` | 3 | 2 | 4 | Non-identity resource matrix |
| `standard_variables_techno2` | 3 | 2 | 3 | Identity control matrix |

In [ ]:
inputs = cc.standard_variables_trad12()

print(f"Actors: {inputs['n']},  Events: {inputs['q']},  Resources: {inputs['m']}")
print("\nDirected interests (y) — rows=actors, cols=events:")
print(inputs['y'])
print("\nResource control (c) — rows=resources, cols=actors:")
print(inputs['c'])

## (Optional) Define your own inputs

Use `cc.setup(n, q, m, y, a, c)` to supply your own matrices.
Matrices are automatically row-normalised before solving.

In [ ]:
# import numpy as np
#
# n, q, m = 3, 4, 4
# y = np.array([[ 0.4,  0.2,  0.1,  0.3],
#               [-0.3,  0.3,  0.2,  0.2],
#               [-0.1, -0.3, -0.5,  0.1]])
# a = np.eye(q)
# c = np.array([[0.3, 0.2, 0.5],
#               [0.4, 0.3, 0.3],
#               [0.4, 0.4, 0.2],
#               [0.25, 0.35, 0.40]])
# inputs = cc.setup(n, q, m, y, a, c)

## 2. Full Coleman analysis

`run_full_analysis` solves for equilibrium actor power `r`, event values `v`,
and resource values `w`, then computes all derived variables in sequence.

In [ ]:
cc.run_full_analysis(inputs)

print(f"Actor power (r):          {inputs['r'].round(4)}")
print(f"Event values (v):         {inputs['v'].round(4)}")
print(f"Outcome probabilities:    {inputs['P_p'].round(4)}")
print(f"Expected collectivity:    {inputs['p_h'].round(4)}")
print(f"Total external power (R): {inputs['R']:.4f}")

In [ ]:
fig1 = cc.draw_power_distribution(inputs)
plt.tight_layout()

In [ ]:
fig2 = cc.draw_event_values(inputs)
plt.tight_layout()

## 3. Coalition analysis

1. **Enumerate** all feasible coalitions (size ≥ 2, majority control of resource 0)
2. **Analyse** each coalition as its own sub-collective
3. **Compute** the transition probability matrix (TPM) — which coalition would each coalition rationally switch to?
4. **Identify** winning (stable, sink) coalitions

In [ ]:
coalitions = cc.feasible_coalitions(inputs)
print(f"Feasible coalitions ({len(coalitions)}): {list(coalitions.keys())}")

In [ ]:
coalition_outputs = cc.coalition_trad(inputs, coalitions)
summary, TPM = cc.optimal_coalition(coalition_outputs)

winning = cc.winning_coalitions(TPM)
winning_names = [name for name, w in zip(coalition_outputs.keys(), winning) if w]
print("Winning (stable) coalitions:", winning_names)

## 4. Coalition visualisation

- **Coalition map** — all transitions, node size = total actor value, highlighted nodes = winning
- **Strongest transitions** — only the dominant outgoing edge per coalition (cleaner view)

In [ ]:
fig3 = cc.draw_coalition_map(coalition_outputs, summary, TPM)
plt.tight_layout()

In [ ]:
fig4 = cc.draw_strongest_transitions(coalition_outputs, summary, TPM)
plt.tight_layout()

## Next steps

- Print or save full results: `cc.print_analysis(inputs)` / `cc.write_analysis(inputs, 'output/', 'run1')`
- Explore the techno-style coalition variant: `cc.coalition_techno_equal_control(inputs, coalitions)`
- Sweep parameters and visualise with `cc.draw_interest_heatmap(data, titles, mask)`